In [1]:
import os
import re
import time
import threading
import numpy as np
import faiss
from flask import Flask, request, jsonify, render_template_string
from werkzeug.utils import secure_filename
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

# --- 1. NUKE GHOST SERVERS ---
print("Clearing port 5000 to prevent conflicts...")
os.system("fuser -k 5000/tcp")
time.sleep(2) # Give the OS a moment to actually release the port

# --- 2. CRASH-PROOF MODEL LOADING ---
if 'models_loaded' not in globals():
    print("Downloading/Loading Models into memory... (This takes a minute)")
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    model = AutoModelForCausalLM.from_pretrained("gpt2")
    llm_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, pad_token_id=tokenizer.eos_token_id)
    models_loaded = True
else:
    print("Models are already in memory! Skipping load to prevent kernel crash.")

# --- 3. INIT VECTOR DB ---
dimension = 384
if 'index' not in globals():
    index = faiss.IndexFlatL2(dimension)
    corpus = []

# --- 4. FLASK SETUP & HTML ---
app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = '/content/uploads'
os.makedirs(app.config['UPLOAD_FOLDER'], exist_ok=True)

html_template = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>RAG Chat Interface</title>
    <style>
        body { font-family: sans-serif; max-width: 800px; margin: 0 auto; padding: 20px; background: #1e1e1e; color: #fff; }
        .container { background: #2d2d2d; padding: 20px; border-radius: 8px; }
        #chat-box { height: 300px; overflow-y: auto; border: 1px solid #444; padding: 10px; margin-bottom: 10px; background: #1e1e1e; }
        .message { margin-bottom: 10px; padding: 8px; border-radius: 4px; }
        .user-msg { background: #005A9E; margin-left: 20%; }
        .bot-msg { background: #333; margin-right: 20%; border: 1px solid #555; }
        input, button { padding: 10px; margin: 5px 0; }
    </style>
</head>
<body>
    <div class="container">
        <h2>1. Upload Knowledge</h2>
        <input type="file" id="file-input" accept=".txt">
        <button onclick="uploadDocument()">Upload & Embed</button>
        <p id="upload-status" style="color: #4CAF50;"></p>

        <h2>2. Chat</h2>
        <div id="chat-box"></div>
        <input type="text" id="question-input" style="width: 70%;" placeholder="Ask a question...">
        <button onclick="askQuestion()">Send</button>
    </div>

    <script>
        let chatHistory = [];

        async function uploadDocument() {
            const fileInput = document.getElementById('file-input');
            if (!fileInput.files[0]) return alert("Select a file.");
            
            document.getElementById('upload-status').innerText = "Processing...";
            const formData = new FormData();
            formData.append("file", fileInput.files[0]);

            const res = await fetch('/upload', { method: 'POST', body: formData });
            const data = await res.json();
            document.getElementById('upload-status').innerText = data.message;
        }

        async function askQuestion() {
            const inputField = document.getElementById('question-input');
            const question = inputField.value.trim();
            if (!question) return;

            appendMessage('user', question);
            inputField.value = '';

            const res = await fetch('/ask', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({ question: question, history: chatHistory })
            });
            const data = await res.json();
            
            appendMessage('bot', data.answer);
            chatHistory.push({ role: "user", content: question });
            chatHistory.push({ role: "assistant", content: data.answer });
            if(chatHistory.length > 6) chatHistory = chatHistory.slice(-6);
        }

        function appendMessage(sender, text) {
            const box = document.getElementById('chat-box');
            const div = document.createElement('div');
            div.className = `message ${sender === 'user' ? 'user-msg' : 'bot-msg'}`;
            div.innerText = text;
            box.appendChild(div);
            box.scrollTop = box.scrollHeight;
        }
    </script>
</body>
</html>
"""

@app.route('/')
def home():
    return render_template_string(html_template)

@app.route('/upload', methods=['POST'])
def upload_file():
    global index, corpus
    if 'file' not in request.files:
        return jsonify({"error": "No file provided"}), 400
        
    file = request.files['file']
    filepath = os.path.join(app.config['UPLOAD_FOLDER'], secure_filename(file.filename))
    file.save(filepath)

    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    
    # Split by any combination of newlines
    chunks = [chunk.strip() for chunk in re.split(r'\n+', text) if len(chunk.strip()) > 20]
    
    # HARD WIPE PREVIOUS DATA
    corpus.clear()
    index.reset() 

    # Embed and add to fresh FAISS index
    embeddings = embed_model.encode(chunks)
    index.add(embeddings.astype('float32'))
    corpus.extend(chunks)

    return jsonify({"message": f"Success! Cleared old data and embedded {len(chunks)} chunks from {file.filename}."})

@app.route('/ask', methods=['POST'])
def ask_question():
    data = request.json
    question = data.get('question', '')
    history = data.get('history', [])

    context_str = ""
    if index.ntotal > 0:
        query_vec = embed_model.encode([question])
        distances, indices = index.search(query_vec.astype('float32'), min(2, index.ntotal))
        retrieved_chunks = [corpus[i] for i in indices[0]]
        context_str = " ".join(retrieved_chunks)

    history_str = ""
    for msg in history[-2:]: # Keep history short
        history_str += f"{msg['role'].capitalize()}: {msg['content']}\n"

    prompt = f"Background Information:\n{context_str}\n\nQuestion: {question}\nAnswer:"

    # Generate the response
    output = llm_pipeline(prompt, max_new_tokens=40, temperature=0.3, truncation=True)
    
    # Extract just the answer portion
    answer = output[0]['generated_text'].split("Answer:")[-1].strip()

    return jsonify({"answer": answer})

# --- 5. SERVER RUNNER ---
def run_flask():
    try:
        # 0.0.0.0 is critical here for VS Code dev tunnels to work
        app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)
    except Exception as e:
        print(f"\n❌ FLASK CRASHED: {e}")

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

time.sleep(2)
if flask_thread.is_alive():
    print("\n✅ --- NEW SERVER RUNNING ON 0.0.0.0:5000 ---")
    print("Click the port 5000 link in your VS Code 'PORTS' tab below.")
else:
    print("\n❌ SERVER FAILED TO START. Please check errors above.")

Clearing port 5000 to prevent conflicts...
Downloading/Loading Models into memory... (This takes a minute)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit



✅ --- NEW SERVER RUNNING ON 0.0.0.0:5000 ---
Click the port 5000 link in your VS Code 'PORTS' tab below.


In [ ]:
# Download Cloudflare's tunnel tool
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

# Start the tunnel to port 5000
import subprocess
import time
import re

print("Starting Cloudflare tunnel...")
process = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:5000'], 
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Read the output to find the unique URL safely
time.sleep(3)
for line in iter(process.stdout.readline, ''):
    if "trycloudflare.com" in line:
        # Use Regex to hunt down the exact URL pattern
        match = re.search(r'https://[a-zA-Z0-9-]+.trycloudflare.com', line)
        if match:
            url = match.group(0)
            print(f"\n✅ YOUR APP IS LIVE AT:\n{url}\n")
            break

cloudflared: Text file busy
Starting Cloudflare tunnel...

✅ YOUR APP IS LIVE AT:
https://api-lecture-hundreds-commerce.trycloudflare.com



INFO:werkzeug:127.0.0.1 - - [06/May/2026 08:45:45] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/May/2026 08:45:45] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [06/May/2026 08:45:57] "POST /upload HTTP/1.1" 200 -
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
INFO:werkzeug:127.0.0.1 - - [06/May/2026 08:46:18] "POST /ask HTTP/1.1" 200 -
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (h